In [7]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [8]:

# ==========================================
# PATH MODEL (ISI SESUAI PUNYA KAMU)
# ==========================================
MODEL_PATHS = {
    "Model-RDC-1.1": "../models/Model-RDC-1.1/model.h5",
    "Model-RDC-2.1": "../models/Model-RDC-2.1/model.h5",
    "Model-RDC-3.1": "../models/Model-RDC-3.1/model.h5",
    "Model-RDC-4.1": "../models/Model-RDC-4.1/model.h5",
}


In [9]:
# ==========================================
# STEP 0: PARAMETERS & PATHS (sesuaikan)
# ==========================================
PROCESSED_DIR = "../dataset_split"  # output: train/ val/ test/ per kelas

IMG_SIZE = (224, 224)   # tuple: target_size untuk flow_from_directory
IMG_SIDE = IMG_SIZE[0]  # integer untuk fungsi make_square
BATCH_SIZE = 16

In [10]:

# ==========================================
# STEP 4: Data generators
# - train: augmentation + rescale
# - val/test: hanya rescale
# ==========================================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

valtest_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
)

val_generator = valtest_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = valtest_datagen.flow_from_directory(
    os.path.join(PROCESSED_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 822 images belonging to 4 classes.
Found 102 images belonging to 4 classes.
Found 105 images belonging to 4 classes.


In [11]:

# ==========================================
# FUNCTION EVALUASI
# ==========================================
def evaluate_model(model, generator):
    generator.reset()

    y_pred = model.predict(generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = generator.classes

    report = classification_report(
        y_true,
        y_pred_classes,
        target_names=list(generator.class_indices.keys()),
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1-score": report["weighted avg"]["f1-score"]
    }


In [12]:
# ==========================================
# LOOP SEMUA MODEL
# ==========================================
results_train = []
results_val = []
results_test = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"\n🚀 Evaluasi {model_name}...")

    model = load_model(model_path)

    # ===== TRAIN =====
    metrics_train = evaluate_model(model, train_generator)
    metrics_train = {k: round(v, 2) for k, v in metrics_train.items()}
    metrics_train["model"] = model_name
    results_train.append(metrics_train)

    # ===== VAL =====
    metrics_val = evaluate_model(model, val_generator)
    metrics_val = {k: round(v, 2) for k, v in metrics_val.items()}
    metrics_val["model"] = model_name
    results_val.append(metrics_val)

    # ===== TEST =====
    metrics_test = evaluate_model(model, test_generator)
    metrics_test = {k: round(v, 2) for k, v in metrics_test.items()}
    metrics_test["model"] = model_name
    results_test.append(metrics_test)


🚀 Evaluasi Model-RDC-1.1...

🚀 Evaluasi Model-RDC-2.1...

🚀 Evaluasi Model-RDC-3.1...

🚀 Evaluasi Model-RDC-4.1...


In [17]:

# ==========================================
# BUAT DATAFRAME
# ==========================================
df_results_train = pd.DataFrame(results_train)

# urutkan kolom biar rapi
df_results_train = df_results_train[["model", "accuracy", "precision", "recall", "f1-score"]]

print("\n📊 Tabel Perbandingan Model Data Train:")
df_results_train



📊 Tabel Perbandingan Model Data Train:


,model,accuracy,precision,recall,f1-score
0,Model-RDC-1.1,0.46,0.44,0.46,0.45
1,Model-RDC-2.1,0.43,0.45,0.43,0.44
2,Model-RDC-3.1,0.38,0.41,0.38,0.39
3,Model-RDC-4.1,0.40,0.43,0.40,0.42


In [18]:

# ==========================================
# BUAT DATAFRAME
# ==========================================
df_results_val = pd.DataFrame(results_val)

# urutkan kolom biar rapi
df_results_val = df_results_val[["model", "accuracy", "precision", "recall", "f1-score"]]

print("\n📊 Tabel Perbandingan Model Data Val:")
df_results_val



📊 Tabel Perbandingan Model Data Val:


,model,accuracy,precision,recall,f1-score
0,Model-RDC-1.1,0.90,0.90,0.90,0.90
1,Model-RDC-2.1,0.91,0.92,0.91,0.91
2,Model-RDC-3.1,0.88,0.92,0.88,0.89
3,Model-RDC-4.1,0.94,0.94,0.94,0.94


In [19]:

# ==========================================
# BUAT DATAFRAME
# ==========================================
df_results_test = pd.DataFrame(results_test)

# urutkan kolom biar rapi
df_results_test = df_results_test[["model", "accuracy", "precision", "recall", "f1-score"]]

print("\n📊 Tabel Perbandingan Model Data Test:")
df_results_test



📊 Tabel Perbandingan Model Data Test:


,model,accuracy,precision,recall,f1-score
0,Model-RDC-1.1,0.88,0.88,0.88,0.87
1,Model-RDC-2.1,0.91,0.93,0.91,0.92
2,Model-RDC-3.1,0.90,0.94,0.90,0.91
3,Model-RDC-4.1,0.93,0.95,0.93,0.94


In [16]:

# ==========================================
# SIMPAN KE CSV
# ==========================================
# save_path = os.path.join(MODEL_DIR, "model_comparison.csv")
# df_results.to_csv(save_path, index=False)

# print(f"\n✅ Disimpan ke: {save_path}")